#Upload and read EEG Excel file

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_rel

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print("EEG dataset loaded successfully.")
display(df.head())
print(df.columns)

#Convert EEG columns to numeric

In [ ]:
numeric_columns = [
    "Attention_Task_NoSmell",
    "Attention_Task_Smell",
    "Relax_Task_NoSmell",
    "Relax_Task_Smell",
    "CogLoad_Task_NoSmell",
    "CogLoad_Task_Smell"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

#Check required columns

In [ ]:
required_columns = [
    "ParticipantID",
    "Attention_Task_NoSmell",
    "Attention_Task_Smell",
    "Relax_Task_NoSmell",
    "Relax_Task_Smell",
    "CogLoad_Task_NoSmell",
    "CogLoad_Task_Smell"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:")
    for col in missing_columns:
        print("-", col)
else:
    print("All required EEG columns are present.")

#Descriptive statistics

In [ ]:
descriptive_results = []

for measure_name, no_smell_col, smell_col in [
    ("Attention during task", "Attention_Task_NoSmell", "Attention_Task_Smell"),
    ("Relaxation during task", "Relax_Task_NoSmell", "Relax_Task_Smell"),
    ("Cognitive load during task", "CogLoad_Task_NoSmell", "CogLoad_Task_Smell")
]:

    descriptive_results.append({
        "EEG_Measure": measure_name,
        "Condition": "No-Smell",
        "Mean": df[no_smell_col].mean(),
        "SD": df[no_smell_col].std(ddof=1)
    })

    descriptive_results.append({
        "EEG_Measure": measure_name,
        "Condition": "Smell",
        "Mean": df[smell_col].mean(),
        "SD": df[smell_col].std(ddof=1)
    })

descriptive_df = pd.DataFrame(descriptive_results)

display(descriptive_df)

descriptive_file = "EEG_DescriptiveStatistics.xlsx" #Table 5.3

descriptive_df.to_excel(
    descriptive_file,
    index=False
)

print(f"{descriptive_file} saved.")

#Shapiro–Wilk normality test

In [ ]:
normality_results = []

for measure_name, no_smell_col, smell_col in [
    ("Attention during task", "Attention_Task_NoSmell", "Attention_Task_Smell"),
    ("Relaxation during task", "Relax_Task_NoSmell", "Relax_Task_Smell"),
    ("Cognitive load during task", "CogLoad_Task_NoSmell", "CogLoad_Task_Smell")
]:

    difference_scores = (
        df[smell_col] - df[no_smell_col]
    ).dropna()

    if len(difference_scores) >= 3:
        stat, p_value = shapiro(difference_scores)

        normality_results.append({
            "EEG_Measure": measure_name,
            "Shapiro_W": stat,
            "p_value": p_value,
            "Normality_Assumption": "Met" if p_value >= 0.05 else "Not met"
        })

    else:
        normality_results.append({
            "EEG_Measure": measure_name,
            "Shapiro_W": np.nan,
            "p_value": np.nan,
            "Normality_Assumption": "Not enough data"
        })

normality_df = pd.DataFrame(normality_results)

display(normality_df)

normality_file = "EEG_NormalityResults.xlsx" #Table F.2

normality_df.to_excel(
    normality_file,
    index=False
)

print(f"{normality_file} saved.")

#Paired-samples t-tests and Cohen’s d

In [ ]:
ttest_results = []

for measure_name, no_smell_col, smell_col in [
    ("Attention during task", "Attention_Task_NoSmell", "Attention_Task_Smell"),
    ("Relaxation during task", "Relax_Task_NoSmell", "Relax_Task_Smell"),
    ("Cognitive load during task", "CogLoad_Task_NoSmell", "CogLoad_Task_Smell")
]:

    paired_data = df[
        ["ParticipantID", no_smell_col, smell_col]
    ].dropna()

    no_smell_values = paired_data[no_smell_col]
    smell_values = paired_data[smell_col]

    difference_scores = smell_values - no_smell_values

    if len(paired_data) >= 2:
        t_stat, p_value = ttest_rel(
            smell_values,
            no_smell_values
        )

        difference_sd = difference_scores.std(ddof=1)

        cohens_d = (
            difference_scores.mean() / difference_sd
            if difference_sd != 0 else np.nan
        )

        ttest_results.append({
            "EEG_Measure": measure_name,
            "NoSmell_Mean": no_smell_values.mean(),
            "NoSmell_SD": no_smell_values.std(ddof=1),
            "Smell_Mean": smell_values.mean(),
            "Smell_SD": smell_values.std(ddof=1),
            "Mean_Difference_Smell_minus_NoSmell": difference_scores.mean(),
            "t_value": t_stat,
            "p_value": p_value,
            "Cohens_d": cohens_d
        })

    else:
        ttest_results.append({
            "EEG_Measure": measure_name,
            "NoSmell_Mean": np.nan,
            "NoSmell_SD": np.nan,
            "Smell_Mean": np.nan,
            "Smell_SD": np.nan,
            "Mean_Difference_Smell_minus_NoSmell": np.nan,
            "t_value": np.nan,
            "p_value": np.nan,
            "Cohens_d": np.nan
        })

ttest_results_df = pd.DataFrame(ttest_results)

display(ttest_results_df)

ttest_file = "EEG_PairedTTestResults.xlsx" #Table 5.4

ttest_results_df.to_excel(
    ttest_file,
    index=False
)

print(f"{ttest_file} saved.")

#Download results

In [ ]:
from google.colab import files

files.download("EEG_PrimaryOutcomes.xlsx")          # needed for master dataset
files.download("EEG_DescriptiveStatistics.xlsx")    # Table 5.3
files.download("EEG_NormalityResults.xlsx")         # Table F.2
files.download("EEG_PairedTTestResults.xlsx")       # Table 5.4